* Desc: Plot cost of living index across the world
* Data: https://www.numbeo.com/cost-of-living/rankings.jsp


In [ ]:
import os
os.environ['PROJ_LIB'] = '${HOME}/miniconda3/share/proj'

import polars as pl
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from config import Continent, european_city_coords, famous_european_cities

In [ ]:
# Create output directory if it doesn't exist
import os
os.makedirs('data/output', exist_ok=True)

In [ ]:
INPUT_FILE = './data/numbeo_cost_of_living_by_cities_2026.csv'
assert INPUT_FILE is not None, f'Input file {INPUT_FILE} not found!' \
                                'Get it from https://www.numbeo.com/cost-of-living/rankings.jsp'

# Load data with Polars
df = pl.read_csv(INPUT_FILE)
print(f'Loaded {len(df)} cities')
df.head()

## Data Cleaning

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.null_count())
print('\nData types:')
print(df.dtypes)

# Check data quality
print('\nFirst few rows:')
df.head(10)

In [ ]:
# Clean city and country columns
# Some cities have state information (e.g., 'Jersey City, NJ, United States')
# Extract just city name and country

df = df.with_columns([
    # Clean city: take first part before comma if comma exists
    pl.col('City').str.strip_chars().str.split(',').list.first().alias('City_Clean'),
    # Clean country: just strip whitespace
    pl.col('Country').str.strip_chars().alias('Country_Clean')
])

print('Cleaned city/country data:')
df.select(['City', 'Country', 'City_Clean', 'Country_Clean']).head(10)

In [ ]:
# Drop rows with missing critical data
df_clean = df.drop_nulls(subset=['Cost_of_Living', 'Cost_of_Living_plus_Rent'])

print(f'Rows after removing missing values: {len(df_clean)} (removed {len(df) - len(df_clean)})')

# Check numeric columns - Polars handles types automatically
numeric_cols = ['Cost_of_Living', 'Rent', 'Cost_of_Living_plus_Rent', 
                'Groceries', 'Restaurant_Price', 'Local_Purchasing_Power']

print('\nStatistics of cleaned data:')
df_clean.select(numeric_cols).describe()

## Filter European Cities

In [ ]:
# Define European countries
european_countries = [
    'Albania', 'Andorra', 'Austria', 'Belarus', 'Belgium', 'Bosnia And Herzegovina',
    'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia',
    'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland',
    'Italy', 'Kosovo', 'Latvia', 'Lithuania', 'Luxembourg',
    'Malta', 'Moldova', 'Monaco', 'Montenegro', 'Netherlands', 'North Macedonia',
    'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'Serbia', 'Slovakia',
    'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Turkey', 'Ukraine', 'United Kingdom'
]

# Filter for European cities
df_europe = df_clean.filter(pl.col('Country_Clean').is_in(european_countries))

print(f'Found {len(df_europe)} European cities out of {len(df_clean)} total cities')
print('\nEuropean cities:')
df_europe.select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent']).sort('Cost_of_Living_plus_Rent', descending=True)

In [ ]:
# Use pre-built city coordinates dataset from config
# Create full location string for better geocoding accuracy
df_europe_clean = df_europe.with_columns([
    (pl.col('City_Clean') + ', ' + pl.col('Country_Clean')).alias('Location')
])

# Convert to Polars DataFrame
city_coords_df = pl.DataFrame({
    'Location': list(european_city_coords.keys()),
    'Latitude': [coord[0] for coord in european_city_coords.values()],
    'Longitude': [coord[1] for coord in european_city_coords.values()]
})

# Merge with df_europe_clean
df_europe_clean_coords = df_europe_clean.join(
    city_coords_df,
    on='Location',
    how='left'
)

# Check how many cities were matched
matched = df_europe_clean_coords.filter(pl.col('Latitude').is_not_null())
unmatched = df_europe_clean_coords.filter(pl.col('Latitude').is_null())

print(f'Matched {len(matched)} out of {len(df_europe_clean)} European cities')
if len(unmatched) > 0:
    print(f'\nUnmatched cities ({len(unmatched)}):')
    print(unmatched['Location'].to_list()[:20])  # Show first 20 unmatched

# Remove rows without coordinates
df_europe_clean_coords = df_europe_clean_coords.drop_nulls(subset=['Latitude', 'Longitude'])
print(f'\nFinal dataset has {len(df_europe_clean_coords)} cities with coordinates')
df_europe_clean_coords.select(['City_Clean', 'Country_Clean', 'Latitude', 'Longitude']).head()

In [ ]:
# Create alias for compatibility with rest of notebook
df_geo = df_europe_clean_coords
print(f'Final dataset has {len(df_geo)} cities with geocodes')

In [ ]:
# Calculate Affordability Index
# Higher value = more affordable (better purchasing power relative to costs)
# This addresses the criticism: shows how affordable each city is for people earning local salaries

df_geo = df_geo.with_columns([
    (pl.col('Local_Purchasing_Power') / pl.col('Cost_of_Living_plus_Rent') * 100).alias('Affordability_Index')
])

print('Affordability Index Statistics:')
print(df_geo.select(['Affordability_Index']).describe())

print('\nMost Affordable Cities (best purchasing power vs costs):')
print(df_geo.sort('Affordability_Index', descending=True).select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent', 'Local_Purchasing_Power', 'Affordability_Index']).head(10))

print('\nLeast Affordable Cities:')
print(df_geo.sort('Affordability_Index', descending=False).select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent', 'Local_Purchasing_Power', 'Affordability_Index']).head(10))

## Limit to famous cities only

In [ ]:
cities_selected = famous_european_cities
df_geo_selected = df_geo.filter(pl.col('City_Clean').is_in(cities_selected))

## Calculate Affordability Index

In [ ]:
# Calculate Affordability Index
# Higher value = more affordable (better purchasing power relative to costs)
# This addresses the criticism: shows how affordable each city is for people earning local salaries

df_geo_selected = df_geo_selected.with_columns([
    (pl.col('Local_Purchasing_Power') / pl.col('Cost_of_Living_plus_Rent') * 100).alias('Affordability_Index')
])

print('Affordability Index Statistics:')
print(df_geo_selected.select(['Affordability_Index']).describe())

print('\nMost Affordable Cities (best purchasing power vs costs):')
print(df_geo_selected.sort('Affordability_Index', descending=True).select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent', 'Local_Purchasing_Power', 'Affordability_Index']).head(10))

print('\nLeast Affordable Cities:')
print(df_geo_selected.sort('Affordability_Index', descending=False).select(['City_Clean', 'Country_Clean', 'Cost_of_Living_plus_Rent', 'Local_Purchasing_Power', 'Affordability_Index']).head(10))

## Affordability Map - European Cities

In [ ]:
# Create Affordability Map for Europe
SHAPE_FILE = './data/natural_earth/ne_10m_admin_0_countries'
RESOLUTION = 'i'  # i = intermediate resolution
N_FIRST_CITIES = 150  # Number of top cities to plot for clarity
FIGSIZE = (20, 12)
CITIES_TO_SHIFT = {
    'Bratislava': (+15000, -15000, ''),
    'Stockholm': (+15000, -15000, ''),
    'Uppsala': (-15000, +15000, 'top'),
    'Vienna': (-15000, +15000, 'top'),
    'Malmo': (+40000, +40000, 'top'),
    'Copenhagen': (-10000, -10000, ''),
    'Espoo': (-20000, +50000, 'top'),
    'Pristina': (-10000, +10000, 'top'),
    'Bologna': (0, +5000, ''),
    'Glasgow': (-10000, +10000, 'top'),
    'Edinburgh': (+10000, -10000, ''),
    'Liverpool': (-10000, +10000, 'top'),
    'Manchester': (+10000, -10000, ''),
    'Cardiff': (-20000, +20000, 'top'),
    'Bristol': (+20000, -20000, ''),
    'Birmingham': (+10000, -10000, ''),
    'Cambridge': (+20000, +20000, 'top'),
    'Lille': (-20000, 0, 'top'),
    'Amsterdam': (0, -20000, 'top'),
    'Rotterdam': (-20000, 0, ''),
    'Padova': (0, -30000, 'top'),
    'Florence': (0, -10000, ''),
    'Malaga': (-10000, +10000, ''),
    'Granada': (+10000, -10000, 'top'),
    'Brno': (0, -10000, 'top'),
    'Dortmund': (0, -10000, 'top'),
    'Heraklion': (0, 0, 'top'),
    'Hamburg': (0, 0, 'top'),
    'Bremen': (-20000, 0, ''),
    'Kaunas': (0, 0, 'top'),
    'Odense': (-10000, -10000, ''),
    'Frankfurt': (+20000, -10000, 'top'),
    'Plzen': (-20000, -10000, ''),
}

# Get Europe configuration
region = Continent.europe
region_llcrnrlon = region['llcrnrlon']
region_llcrnrlat = region['llcrnrlat'] - 1.0  # Adjust to southern boundary
region_urcrnrlon = region['urcrnrlon'] - 12.0 # Adjust to eastern boundary
region_urcrnrlat = region['urcrnrlat']
region_lat0 = region['lat_0']
region_lon0 = region['lon_0']
region_lat1 = region['lat_1']
region_lat2 = region['lat_2']

# Create figure
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(111)

# Create Basemap for Europe
m = Basemap(
    llcrnrlon=region_llcrnrlon,
    llcrnrlat=region_llcrnrlat,
    urcrnrlon=region_urcrnrlon,
    urcrnrlat=region_urcrnrlat,
    lat_0=region_lat0,
    lon_0=region_lon0,
    lat_1=region_lat1,
    lat_2=region_lat2,
    projection='lcc',
    resolution=RESOLUTION,
    area_thresh=1000,
)

# Draw map boundary and features
m.drawmapboundary(fill_color='lightblue', linewidth=0.5)
m.fillcontinents(color='wheat', lake_color='lightblue')
m.drawcountries(linewidth=1.5, color='darkgray')
m.drawcoastlines(linewidth=0.5)

# Read shapefile for political boundaries
m.readshapefile(SHAPE_FILE, 'countries', linewidth=1, color='gray')

# Get data sorted by affordability
df_sorted = df_geo_selected.sort('Affordability_Index', descending=True)
lats = df_sorted['Latitude'].to_list()
lons = df_sorted['Longitude'].to_list()
affordability = df_sorted['Affordability_Index'].to_list()
cities = df_sorted['City_Clean'].to_list()

# Drop None values from lats, lons, costs, cities
valid_data = [(lat, lon, aff, city) for lat, lon, aff, city in zip(lats, lons, affordability, cities)
              if lat is not None and lon is not None and aff is not None and city is not None]
lats, lons, affordability, cities = zip(*valid_data)

# Limit to first N cities for clarity
lats = lats[:N_FIRST_CITIES]
lons = lons[:N_FIRST_CITIES]
affordability = affordability[:N_FIRST_CITIES]
cities = cities[:N_FIRST_CITIES]

# Convert coordinates to map projection
x, y = m(lons, lats)

# Scale circle sizes by affordability
sizes = [(a ** 1.1) * 1.1 for a in affordability]

# Apply shifts to coordinates for cities in CITIES_TO_SHIFT
x_shifted = []
y_shifted = []
for xi, yi, city in zip(x, y, cities):
    if city in CITIES_TO_SHIFT:
        shift_x, shift_y, _ = CITIES_TO_SHIFT[city]
        x_shifted.append(xi + shift_x)
        y_shifted.append(yi + shift_y)
    else:
        x_shifted.append(xi)
        y_shifted.append(yi)

# Create scatter plot with circles (reverse colormap: green=good, red=bad)
scatter = m.scatter(x_shifted, y_shifted, s=sizes, alpha=0.7, c=affordability,
                    cmap='RdYlBu', edgecolors='black', linewidth=1.5,
                    zorder=5)

# Add value labels inside circles
for xi, yi, aff, city in zip(x_shifted, y_shifted, affordability, cities):
    # Affordability value
    plt.text(xi, yi, f'{aff:.0f}',
             ha='center', va='center', fontsize=7, fontweight='bold',
             color='black', zorder=6)

    # City name label with white background for readability
    if city in CITIES_TO_SHIFT:
        _, _, position = CITIES_TO_SHIFT[city]
        if position == 'top':
            plt.text(xi, yi + 70000, city,
                     ha='center', va='top', fontsize=6, zorder=6, style='italic',
                    color='darkblue',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))
        else:
            plt.text(xi, yi - 30000, city,
                     ha='center', va='top', fontsize=6, zorder=6, style='italic',
                    color='darkblue',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))
    else:
        plt.text(xi, yi - 30000, city,
                ha='center', va='top', fontsize=6, zorder=6, style='italic',
                color='darkblue',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Affordability Index (Higher = Better)', fontsize=12)

# Title for map
plt.title('Affordability in European Cities (2026)\nLocal Purchasing Power vs Cost of Living',
          fontsize=18, fontweight='bold', pad=20)

# Add formula annotation
formula_text = r'Affordability = $\frac{Local\ Purchasing\ Power}{Cost\ of\ Living} \times 100$'
ax.text(0.16, 0.95, formula_text,
        transform=ax.transAxes, fontsize=14, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.savefig('data/output/europe_affordability_map.png', dpi=300, bbox_inches='tight')
print('Saved map as data/output/europe_affordability_map.png')
plt.show()


## 2-Axis Scatter Plot: Cost vs Purchasing Power

In [ ]:
N_FIRST_CITIES = 100  # Number of top cities to plot for clarity
N_TOP = 20
N_BOTTOM = 20
CITIES_TO_SHIFT = {
    'Graz': (5, -5),
    'Rome': (5, -5)
}

# Create 2-axis scatter plot: Cost of Living vs Purchasing Power
fig, ax = plt.subplots(figsize=(10, 10))

# Get data
costs = df_geo_selected['Cost_of_Living_plus_Rent'].to_list()
purchasing_power = df_geo_selected['Local_Purchasing_Power'].to_list()
affordability = df_geo_selected['Affordability_Index'].to_list()
cities = df_geo_selected['City_Clean'].to_list()

# Limit to first N cities for clarity
data_tuples = list(zip(costs, purchasing_power, affordability, cities))
data_tuples = data_tuples[:N_FIRST_CITIES]
costs_, purchasing_power_, affordability_, cities_ = zip(*data_tuples)

# Take only given top and bottom affordability cities for scatter plot
costs = []
purchasing_power = []
affordability = []
for cost, power, aff in zip(costs_, purchasing_power_, affordability_):
    if aff in (sorted(affordability_, reverse=True)[:N_TOP] + sorted(affordability_)[:N_BOTTOM]):
        costs.append(cost)
        purchasing_power.append(power)
        affordability.append(aff)

# Same circle sizes for affordability
sizes = [a for a in affordability]

# Create scatter plot with color by affordability
scatter = ax.scatter(costs, purchasing_power, s=sizes, alpha=0.6, 
                     c=affordability, cmap='RdYlBu', edgecolors='black', 
                     linewidth=1.5, vmin=50, vmax=350)

# Add city labels for top/bottom affordability cities
df_labeled = df_geo_selected.sort('Affordability_Index', descending=True)
top_cities = df_labeled.head(N_TOP)
bottom_cities = df_labeled.tail(N_BOTTOM)
labeled_cities = pl.concat([top_cities, bottom_cities])

for row in labeled_cities.iter_rows(named=True):
    xytext=(5, 5)
    if row['City_Clean'] in CITIES_TO_SHIFT:
        xytext = CITIES_TO_SHIFT[row['City_Clean']]
    ax.annotate(row['City_Clean'], 
                (row['Cost_of_Living_plus_Rent'], row['Local_Purchasing_Power']),
                xytext=xytext, textcoords='offset points',
                fontsize=7, alpha=0.7,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                         edgecolor='gray', alpha=0.7))

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Affordability Index\n(Higher = Better)', fontsize=12)

# Labels and title
ax.set_xlabel('Cost of Living + Rent Index (Higher = More Expensive)', fontsize=14, fontweight='bold')
ax.set_ylabel('Local Purchasing Power (Higher = Better)', fontsize=14, fontweight='bold')
ax.set_title('Cost of Living vs Purchasing Power in European Cities (2026)', 
             fontsize=16, fontweight='bold', pad=20)

# Set axis limits
# ax.set_xlim(min(costs) * 0.9, max(costs) * 1.1)
ax.set_xlim(min(costs) * 0.95, max(costs) * 1.05)
ax.set_ylim(min(purchasing_power) * 0.95, max(purchasing_power) * 1.05)

# # Set axis limits
# min_lim = min(min(costs) * 0.95, min(purchasing_power) * 0.95)
# max_lim = max(max(costs) * 1.05, max(purchasing_power) * 1.05)
# ax.set_xlim(min_lim, max_lim)
# ax.set_ylim(min_lim, max_lim)

# Grid
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)

plt.tight_layout()
plt.savefig('data/output/europe_cost_vs_income_scatter.png', dpi=300, bbox_inches='tight')
print('Saved scatter plot as data/output/europe_cost_vs_income_scatter.png')
plt.show()